# Notebook 00 — Environment Setup & Hardware Assessment

**Assignment 1B — Enterprise Domain LLM**  
**Author:** Abhinav Mandloi  

This notebook verifies the hardware configuration, Python environment, and all required packages before any experimentation begins. All subsequent notebooks depend on the conditions confirmed here.

---
## Objectives
1. Inspect system hardware (CPU, RAM, GPU/MPS/CUDA)
2. Report all package versions
3. Verify MPS / CUDA availability for PyTorch
4. Produce a formatted hardware assessment table

In [ ]:
import os
import sys
import platform
import subprocess
import psutil
import torch

print('=' * 60)
print('SYSTEM INFORMATION')
print('=' * 60)
print(f'OS            : {platform.system()} {platform.release()} ({platform.version()})')
print(f'Machine       : {platform.machine()}')
print(f'Processor     : {platform.processor()}')
print(f'Python        : {sys.version}')
print(f'Python path   : {sys.executable}')

# CPU info
cpu_count_logical  = psutil.cpu_count(logical=True)
cpu_count_physical = psutil.cpu_count(logical=False)
print(f'\nCPU logical   : {cpu_count_logical}')
print(f'CPU physical  : {cpu_count_physical}')

# RAM
ram = psutil.virtual_memory()
print(f'Total RAM     : {ram.total / 1e9:.2f} GB')
print(f'Available RAM : {ram.available / 1e9:.2f} GB')

# Disk
disk = psutil.disk_usage('/')
print(f'Disk total    : {disk.total / 1e9:.1f} GB')
print(f'Disk free     : {disk.free / 1e9:.1f} GB')

## 1.1 GPU / Accelerator Detection

In [ ]:
print('=' * 60)
print('GPU / ACCELERATOR')
print('=' * 60)
print(f'PyTorch version  : {torch.__version__}')

# CUDA
cuda_available = torch.cuda.is_available()
print(f'CUDA available   : {cuda_available}')
if cuda_available:
    print(f'  CUDA version   : {torch.version.cuda}')
    print(f'  Device count   : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB')

# MPS (Apple Silicon)
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f'MPS available    : {mps_available}')
if mps_available:
    print('  MPS (Metal Performance Shaders) on Apple Silicon is available.')
    print('  BFloat16 is supported on MPS (PyTorch >= 2.x).')
    print('  NOTE: BitsAndBytes 4-bit (NF4) quantization is NOT supported on MPS.')

# Select device
if cuda_available:
    DEVICE = 'cuda'
elif mps_available:
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'\nSelected device  : {DEVICE}')

## 1.2 Package Version Report

In [ ]:
import importlib
import pandas as pd

packages = [
    'torch', 'transformers', 'datasets', 'accelerate',
    'peft', 'trl', 'huggingface_hub', 'tokenizers',
    'sentencepiece', 'fitz',  # fitz = PyMuPDF
    'langdetect', 'sklearn', 'matplotlib',
    'pandas', 'numpy', 'pyarrow', 'psutil', 'bitsandbytes'
]

rows = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'n/a')
        status = '✅'
    except ImportError:
        version = 'NOT INSTALLED'
        status = '❌'
    rows.append({'Package': pkg, 'Version': version, 'Status': status})

df_pkgs = pd.DataFrame(rows)
print(df_pkgs.to_string(index=False))

## 1.3 Hardware Assessment Table

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

assessment = [
    ('CPU', 'Apple M4', 'arm64', 'Available', '✅'),
    ('RAM', '24 GB Unified', '—', 'Available', '✅'),
    ('MPS (Apple GPU)', 'Metal Performance Shaders', 'arm64', 'Available', '✅'),
    ('BFloat16 on MPS', 'PyTorch >= 2.x', '—', 'Supported', '✅'),
    ('CUDA', 'NVIDIA GPU', 'x86_64', 'NOT available', '❌'),
    ('4-bit NF4 BnB on MPS', 'BitsAndBytes', '—', 'NOT supported', '❌'),
    ('4-bit NF4 BnB on CUDA', 'BitsAndBytes', 'cuda', 'Needs Colab/cloud', '⚠️'),
]

df_hw = pd.DataFrame(assessment, columns=['Component', 'Detail', 'Architecture', 'Status', 'Flag'])
print('Hardware Assessment:')
print(df_hw.to_string(index=False))

print('\n--- DEVICE CONFIGURATION ---')
print(f'Training / Inference device  : {DEVICE}')
print(f'Precision                    : bfloat16 (MPS) or float32 (CPU fallback)')
print(f'QLoRA 4-bit on this machine  : NOT POSSIBLE (MPS limitation)')
print(f'LoRA fine-tuning (bfloat16)  : POSSIBLE on MPS')
print(f'Speculative decoding         : POSSIBLE on MPS')

## 1.4 Quick Tensor Smoke Test

In [ ]:
# Verify bfloat16 tensor creation on MPS
device = torch.device(DEVICE)
try:
    x = torch.ones(4, 4, dtype=torch.bfloat16, device=device)
    y = x @ x.T
    print(f'BF16 matmul on {DEVICE}: {y.shape}, dtype={y.dtype}  ✅')
except Exception as e:
    print(f'BF16 matmul failed: {e}  ❌')

# Float32
try:
    x2 = torch.ones(4, 4, dtype=torch.float32, device=device)
    y2 = x2 @ x2.T
    print(f'FP32 matmul on {DEVICE}: {y2.shape}, dtype={y2.dtype}  ✅')
except Exception as e:
    print(f'FP32 matmul failed: {e}  ❌')

print('\nEnvironment setup complete. Proceed to Notebook 01.')